<!-- paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 참고 구현입니다. 필요한 한 줄만 확인하고 다시 실습 창으로 돌아가세요.

# 17. Embedding과 Vector Index — 직접 만들고 평가하기

외부 API와 다운로드 없이 `knowledge_base.jsonl`을
**문서 검증 → overlap chunking → sparse/dense embedding → 정규화 →
NumPy vector index → 저장/로드 → 검색 평가**까지 연결합니다.
작은 데이터이므로 RTX GPU가 없어도 실행되며,
GPU가 있으면 PyTorch dense encoder 학습에 자동으로 사용합니다.


In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 두 창 학습법

왼쪽에는 `notebooks/exercises/17_...ipynb`, 오른쪽에는 `notebooks/solutions/17_...ipynb`를 여세요.
먼저 왼쪽의 `TODO`를 직접 채우고 작은 입력으로 결과를 예상한 뒤 오른쪽 셀과 비교합니다.
정답을 복사하기보다 **입력 shape, 정규화 축, id/metadata 매핑**을 소리 내어 설명하는 것이 목표입니다.


## 전체 흐름

```text
JSONL → schema/id 검증 → overlap chunks
  ├─ char TF-IDF sparse vectors ─┐
  └─ tiny PyTorch dual encoder ──┼→ L2 normalize → cosine top-k
                                └→ metadata + vector 저장/로드 → Recall@k / MRR
```

실무의 FAISS/Qdrant/pgvector도 핵심 계약은 같습니다. `vector row i`가 어느 문서·chunk인지 잃지 않는 것이 특히 중요합니다.


In [ ]:
import json, math, random, tempfile
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from llm_engineering_lab.acceleration import get_accelerator


def find_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data" / "raw" / "knowledge_base.jsonl").exists():
            return candidate
    raise RuntimeError("코딩연습 폴더 또는 그 하위에서 실행하세요.")


ROOT = find_root()
KB_PATH = ROOT / "data" / "raw" / "knowledge_base.jsonl"
SEED = 17
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print(ACCELERATOR.summary(), "| knowledge_base=", KB_PATH)

## 1) JSONL 로드와 데이터 계약

JSONL은 한 줄이 한 문서입니다. 필수 필드, 빈 문자열, 중복 id를 색인 전에 막아야 나중의 row/id 불일치를 피할 수 있습니다.
파일에는 UTF-8 BOM이 있을 수 있으므로 `utf-8-sig`로 읽습니다.


In [ ]:
REQUIRED_FIELDS = {"id", "title", "category", "content", "keywords"}


def load_and_validate_documents(path: Path) -> list[dict]:
    documents = []
    with path.open("r", encoding="utf-8-sig") as handle:
        for line_no, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            missing = REQUIRED_FIELDS - record.keys()
            if missing:
                raise ValueError(f"line {line_no}: missing={sorted(missing)}")
            if any(not str(record[key]).strip() for key in ("id", "title", "content")):
                raise ValueError(f"line {line_no}: empty required value")
            if not isinstance(record["keywords"], list):
                raise TypeError(f"line {line_no}: keywords must be list")
            documents.append(record)
    ids = [document["id"] for document in documents]
    if len(ids) != len(set(ids)):
        raise ValueError("document id must be unique")
    return documents


documents = load_and_validate_documents(KB_PATH)
assert len(documents) == 12

In [ ]:
document_frame = pd.DataFrame(
    {
        "id": d["id"],
        "title": d["title"],
        "category": d["category"],
        "content_chars": len(d["content"]),
        "keyword_count": len(d["keywords"]),
    }
    for d in documents
)
assert document_frame["id"].is_unique
display(document_frame)

## 2) overlap chunking과 metadata

긴 문서를 일정 길이로 자르면 query와 관련된 정보가 희석되지 않습니다. 경계에 걸린 문장을 보존하려고 이전 chunk 끝 일부를
다음 chunk에 겹칩니다. `chunk_id`, `source_id`, 문자 범위, 제목/카테고리를 함께 보존합니다.


In [ ]:
@dataclass(frozen=True)
class Chunk:
    chunk_id: str
    source_id: str
    title: str
    category: str
    text: str
    start: int
    end: int
    keywords: tuple[str, ...]

    @property
    def searchable_text(self) -> str:
        return " ".join((self.title, self.category, *self.keywords, self.text))


def chunk_documents(
    records: list[dict], chunk_size: int = 120, overlap: int = 30
) -> list[Chunk]:
    if chunk_size <= 0 or not 0 <= overlap < chunk_size:
        raise ValueError("chunk_size > 0 and 0 <= overlap < chunk_size 이어야 합니다.")
    chunks = []
    for record in records:
        text = record["content"].strip()
        start, part = 0, 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            chunks.append(
                Chunk(
                    chunk_id=f"{record['id']}::c{part:03d}",
                    source_id=record["id"],
                    title=record["title"],
                    category=record["category"],
                    text=text[start:end],
                    start=start,
                    end=end,
                    keywords=tuple(record["keywords"]),
                )
            )
            if end == len(text):
                break
            start, part = end - overlap, part + 1
    return chunks

In [ ]:
chunks = chunk_documents(documents, chunk_size=120, overlap=30)
assert len({c.chunk_id for c in chunks}) == len(chunks)
assert all(0 <= c.start < c.end and c.end - c.start <= 120 for c in chunks)
assert {c.source_id for c in chunks} == {d["id"] for d in documents}
chunk_frame = pd.DataFrame(asdict(c) for c in chunks)
display(chunk_frame[["chunk_id", "source_id", "start", "end", "text"]].head(8))
print("documents=", len(documents), "chunks=", len(chunks))

## 3) sparse embedding: 문자 n-gram TF-IDF

한국어 형태소 분석기 없이도 문자 2~5 gram은 오류 코드, 합성어, 조사 변화에 비교적 강합니다. scikit-learn의 TF-IDF는 기본적으로
각 row를 L2 정규화하므로 내적이 cosine similarity가 됩니다. sparse matrix이므로 0을 저장하지 않습니다.


In [ ]:
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)
sparse_matrix = tfidf.fit_transform([chunk.searchable_text for chunk in chunks]).tocsr()
assert sparse_matrix.shape[0] == len(chunks)
sparsity = 1.0 - sparse_matrix.nnz / np.prod(sparse_matrix.shape)
print(
    "shape=",
    sparse_matrix.shape,
    "nnz=",
    sparse_matrix.nnz,
    "sparsity=",
    round(sparsity, 4),
)

In [ ]:
def sparse_search(query: str, top_k: int = 3) -> list[dict]:
    if not query.strip() or top_k <= 0:
        return []
    query_vector = tfidf.transform([query])
    scores = (sparse_matrix @ query_vector.T).toarray().ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return [
        {
            "rank": rank,
            "score": float(scores[i]),
            "chunk_id": chunks[i].chunk_id,
            "source_id": chunks[i].source_id,
            "title": chunks[i].title,
        }
        for rank, i in enumerate(order, start=1)
    ]

In [ ]:
query = "카드에서 동일한 금액이 두 번 결제됐어요"
sparse_results = sparse_search(query, top_k=3)
assert "kb-003" in {row["source_id"] for row in sparse_results}
display(pd.DataFrame(sparse_results).round({"score": 3}))

## 4) dense embedding: 작은 PyTorch dual encoder

query encoder와 document encoder가 각각 문자열을 고정 길이 벡터로
바꿉니다. 여기서는 문자 embedding의 masked mean과 작은 projection을
사용합니다. 같은 batch의 다른 문서를 negative로 두는 in-batch
contrastive loss를 학습합니다. 12개 예시를 외우는 교육용 모델이며
범용 sentence embedding 모델이 아닙니다.


In [ ]:
QUERY_PAIRS = [
    ("비밀번호를 잊어버렸어요", "kb-001"),
    ("로그인 실패로 계정이 잠겼어요", "kb-002"),
    ("같은 주문이 두 번 결제됐어요", "kb-003"),
    ("구독 자동 갱신을 해지하고 싶어요", "kb-004"),
    ("환불이 카드에 언제 반영되나요", "kb-005"),
    ("운송장 배송 조회가 멈췄어요", "kb-006"),
    ("출고 뒤 배송지 주소를 바꿀 수 있나요", "kb-007"),
    ("파손된 상품을 교환하고 싶어요", "kb-008"),
    ("API 호출에서 401 인증 오류가 납니다", "kb-009"),
    ("429 rate limit 재시도 방법", "kb-010"),
    ("서버 5xx 장애를 확인하는 절차", "kb-011"),
    ("개인정보 내보내기와 계정 삭제", "kb-012"),
]
document_by_id = {d["id"]: d for d in documents}
train_document_texts = [
    " ".join(
        (
            document_by_id[i]["title"],
            *document_by_id[i]["keywords"],
            document_by_id[i]["content"],
        )
    )
    for _, i in QUERY_PAIRS
]
all_texts = (
    [q for q, _ in QUERY_PAIRS]
    + train_document_texts
    + [c.searchable_text for c in chunks]
)
characters = sorted(set("".join(all_texts)))
char_to_id = {char: index + 2 for index, char in enumerate(characters)}
char_to_id["<PAD>"] = 0
char_to_id["<UNK>"] = 1
print("vocabulary=", len(char_to_id))

In [ ]:
def batch_char_ids(
    texts: list[str], max_length: int = 180
) -> tuple[torch.Tensor, torch.Tensor]:
    rows = [[char_to_id.get(ch, 1) for ch in text[:max_length]] for text in texts]
    width = max(1, max(len(row) for row in rows))
    ids = torch.zeros((len(rows), width), dtype=torch.long)
    for row_index, row in enumerate(rows):
        ids[row_index, : len(row)] = torch.tensor(row, dtype=torch.long)
    return ids, ids.ne(0)


train_queries = [q for q, _ in QUERY_PAIRS]
query_ids, query_mask = batch_char_ids(train_queries)
document_ids, document_mask = batch_char_ids(train_document_texts)
assert query_ids.ndim == 2 and query_mask.dtype == torch.bool

In [ ]:
class TinyDualEncoder(torch.nn.Module):
    def __init__(self, vocab_size: int, hidden: int = 64, output_dim: int = 48):
        super().__init__()
        self.query_embedding = torch.nn.Embedding(vocab_size, hidden, padding_idx=0)
        self.document_embedding = torch.nn.Embedding(vocab_size, hidden, padding_idx=0)
        self.query_projection = torch.nn.Linear(hidden, output_dim)
        self.document_projection = torch.nn.Linear(hidden, output_dim)

    def encode(self, ids: torch.Tensor, mask: torch.Tensor, tower: str) -> torch.Tensor:
        embedding = (
            self.query_embedding if tower == "query" else self.document_embedding
        )
        projection = (
            self.query_projection if tower == "query" else self.document_projection
        )
        token_vectors = embedding(ids)
        weights = mask.unsqueeze(-1).to(token_vectors.dtype)
        pooled = (token_vectors * weights).sum(1) / weights.sum(1).clamp_min(1.0)
        return torch.nn.functional.normalize(projection(pooled), p=2, dim=-1)


model = ACCELERATOR.move(TinyDualEncoder(len(char_to_id)))

## 5) contrastive training

정답 행렬의 대각선이 positive입니다.
`query_vectors @ document_vectors.T / temperature`는 `[B, B]` logits이고,
target은 `[0, 1, ..., B-1]`입니다. 작은 temperature는 점수 차이를 확대합니다.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-2, weight_decay=1e-4)
targets = torch.arange(len(QUERY_PAIRS), device=DEVICE)
losses = []
q_ids, q_mask, d_ids, d_mask = ACCELERATOR.move(
    query_ids, query_mask, document_ids, document_mask
)
model.train()
for step in range(160):
    optimizer.zero_grad(set_to_none=True)
    q_vectors = model.encode(q_ids, q_mask, "query")
    d_vectors = model.encode(d_ids, d_mask, "document")
    logits = q_vectors @ d_vectors.T / 0.08
    loss = (
        torch.nn.functional.cross_entropy(logits, targets)
        + torch.nn.functional.cross_entropy(logits.T, targets)
    ) / 2
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))
assert losses[-1] < losses[0]
print("first/final loss=", round(losses[0], 4), round(losses[-1], 4))

## 6) L2 normalization과 cosine similarity

모든 vector를 unit length로 만들면 cosine similarity는 단순 내적입니다. query와 index에서 서로 다른 정규화 규칙을 사용하면
ranking이 조용히 망가집니다. vector shape와 norm을 계약으로 검증하세요.


In [ ]:
@torch.inference_mode()
def dense_encode(texts: list[str], tower: str) -> np.ndarray:
    ids, mask = batch_char_ids(texts)
    model.eval()
    ids, mask = ACCELERATOR.move(ids, mask)
    vectors = model.encode(ids, mask, tower)
    return vectors.cpu().numpy().astype(np.float32)


chunk_vectors = dense_encode([c.searchable_text for c in chunks], "document")
norms = np.linalg.norm(chunk_vectors, axis=1)
assert chunk_vectors.shape == (len(chunks), 48)
assert np.allclose(norms, 1.0, atol=1e-5)
print("vectors=", chunk_vectors.shape, "norm range=", (norms.min(), norms.max()))

## 7) NumPy brute-force vector index

이 데이터에서는 모든 vector와 query의 내적을 계산해도 충분히 빠릅니다. index는 vector와 metadata 길이, dimension,
고유 chunk id를 검증하고 top-k에서 row index를 metadata로 되돌립니다.


In [ ]:
class NumpyVectorIndex:
    def __init__(self, vectors: np.ndarray, metadata: list[dict]):
        vectors = np.asarray(vectors, dtype=np.float32)
        if vectors.ndim != 2 or len(vectors) != len(metadata):
            raise ValueError("vectors must be [N,D] and align with metadata")
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        if np.any(norms == 0):
            raise ValueError("zero vector cannot be indexed")
        ids = [row["chunk_id"] for row in metadata]
        if len(ids) != len(set(ids)):
            raise ValueError("chunk_id must be unique")
        self.vectors = vectors / norms
        self.metadata = list(metadata)

    def search(self, query_vector: np.ndarray, top_k: int = 3) -> list[dict]:
        query_vector = np.asarray(query_vector, dtype=np.float32).reshape(-1)
        if query_vector.shape[0] != self.vectors.shape[1] or top_k <= 0:
            raise ValueError("query dimension mismatch or invalid top_k")
        query_vector = query_vector / max(float(np.linalg.norm(query_vector)), 1e-12)
        scores = self.vectors @ query_vector
        order = np.argsort(scores)[::-1][: min(top_k, len(scores))]
        return [
            {**self.metadata[i], "rank": rank, "score": float(scores[i])}
            for rank, i in enumerate(order, start=1)
        ]

    def save(self, folder: Path) -> None:
        folder.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(folder / "vectors.npz", vectors=self.vectors)
        (folder / "metadata.json").write_text(
            json.dumps(self.metadata, ensure_ascii=False, indent=2), encoding="utf-8"
        )

    @classmethod
    def load(cls, folder: Path):
        vectors = np.load(folder / "vectors.npz", allow_pickle=False)["vectors"]
        metadata = json.loads((folder / "metadata.json").read_text(encoding="utf-8"))
        return cls(vectors, metadata)

## 8) index 저장·로드와 id mapping 회귀 테스트

실제 서비스에서는 offline indexer가 만든 artifact를 online retriever가 읽습니다. 여기서는 임시 폴더에 저장해 원본 학습 폴더를
더럽히지 않습니다. 재로드 전후 점수·id가 같아야 합니다.


In [ ]:
metadata = [asdict(chunk) for chunk in chunks]
dense_index = NumpyVectorIndex(chunk_vectors, metadata)
query_vector = dense_encode(["429 오류의 재시도 방법"], "query")[0]
before = dense_index.search(query_vector, top_k=3)

with tempfile.TemporaryDirectory() as temporary_directory:
    index_path = Path(temporary_directory) / "dense_index"
    dense_index.save(index_path)
    restored_index = NumpyVectorIndex.load(index_path)
    after = restored_index.search(query_vector, top_k=3)
assert [r["chunk_id"] for r in before] == [r["chunk_id"] for r in after]
assert np.allclose([r["score"] for r in before], [r["score"] for r in after])
display(pd.DataFrame(after)[["rank", "source_id", "chunk_id", "score", "title"]])

## 9) retrieval 평가: Recall@k와 MRR

chunk 검색 결과를 원문 `source_id` 기준으로 평가합니다. Recall@k는 정답 문서가 top-k 안에 있는 비율이고,
MRR은 첫 정답 순위의 역수 평균이라 상위 순위를 더 강하게 보상합니다. generation 평가와 분리해야 실패 지점을 찾을 수 있습니다.


In [ ]:
EVAL_CASES = QUERY_PAIRS + [
    ("인증 링크가 이메일로 오지 않아요", "kb-001"),
    ("택배 위치가 48시간 동안 그대로예요", "kb-006"),
    ("Retry-After와 지수 백오프가 궁금해요", "kb-010"),
]


def evaluate(search_fn, cases, top_k: int = 3) -> dict:
    hits, reciprocal_ranks = 0, []
    for eval_query, expected_id in cases:
        results = search_fn(eval_query, top_k)
        source_ids = [row["source_id"] for row in results]
        hits += expected_id in source_ids
        reciprocal_ranks.append(
            1 / (source_ids.index(expected_id) + 1)
            if expected_id in source_ids
            else 0.0
        )
    return {"recall_at_k": hits / len(cases), "mrr": float(np.mean(reciprocal_ranks))}

In [ ]:
def dense_search(text: str, top_k: int = 3) -> list[dict]:
    return dense_index.search(dense_encode([text], "query")[0], top_k=top_k)


comparison = pd.DataFrame(
    {"retriever": name, "k": k, **evaluate(search_fn, EVAL_CASES, k)}
    for name, search_fn in (
        ("sparse-char-tfidf", sparse_search),
        ("tiny-dense", dense_search),
    )
    for k in (1, 3, 5)
)
assert comparison["recall_at_k"].between(0, 1).all()
display(comparison.round(3))

## 10) 결과를 해석하는 법

작은 dense 모델은 훈련 query를 잘 외우지만 새로운 표현에는 약할 수
있습니다. TF-IDF는 정확한 오류 코드와 단어가 강하고, dense model은
표현이 달라도 가까운 의미를 목표로 합니다. 현업에서는 두 score를
calibration하거나 reciprocal-rank fusion으로 합치고,
카테고리/권한/버전 metadata filter를 적용합니다. 모델의 train metric보다 **고정된 미관측 retrieval 평가셋**을 우선하세요.


## 선택 확장: FAISS

데이터가 수십만 개를 넘으면 `IndexFlatIP`(정확 검색)나 HNSW/IVF(근사 검색)를 고려할 수 있습니다. FAISS를 쓸 때도 입력을
`float32`와 L2 unit vector로 맞추고, FAISS row id ↔ metadata 저장소 id 매핑을 별도로 보존하세요.
**이 실습은 FAISS가 없어도 완전히 동작**하며 NumPy 결과가 정확 검색 기준선입니다.


## 직접 해볼 실험

1. `chunk_size/overlap`을 바꾸고 chunk 수, Recall@3, context 중복을 기록하세요.
2. dense output dimension을 16/48/128로 바꾸고 속도와 MRR을 비교하세요.
3. 평가 query를 직접 20개 추가하되 train query와 분리하세요.
4. sparse와 dense 순위를 RRF로 합치고 개선된 query와 악화된 query를 각각 설명하세요.
5. metadata에서 `category` filter를 넣되 filter 전후에 top-k 정의가 어떻게 달라지는지 검증하세요.

완료 기준: embedding 값 자체보다 **데이터 계약 → row/id 매핑 → 평가 → 저장/로드 일관성**을 설명할 수 있습니다.
